In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# 新内容、新编辑位置：局部同步复现

新内容固定为静止镜头、单只红色小帆船缓慢驶过湖面；seed=20260916，编辑源帧固定90（零基）。不扫描位置，不按结果换内容。

冻结已有支持、margin1、四相位状态编码、评分权重和候选集合。九条接收视频共用一个新生成终端：1次生成、计划100次Transformer前向、3次VAE解码、36次VAE重编码。实际尝试/完成数持续落盘。

同一观察、同一局部路径和事件代价比较 local_matched、local_without_update、local_state；保留两种全局对照。AISB与生成Flow写入位置均不变。

请选择GPU运行时并依次运行所有单元。此入口已通过CPU及结构检查，真实生成仍由你启动。


## 拉取固定源码与安装

源码 SHA：`4a57059de7eab8be4529d2d37e0e5b31fad75907`。复用已跑通的依赖、加载与子进程路径。

In [ ]:
from pathlib import Path
import sys,subprocess
URL='https://github.com/RICHAAARC/SC-SSTW.git'; REF='4a57059de7eab8be4529d2d37e0e5b31fad75907'; SOURCE=Path('/content/wan_state_clock_replication_source')
if SOURCE.exists():
    if subprocess.check_output(['git','-C',str(SOURCE),'remote','get-url','origin'],text=True).strip()!=URL: raise RuntimeError('unexpected origin')
else:
    subprocess.run(['git','init',str(SOURCE)],check=True); subprocess.run(['git','-C',str(SOURCE),'remote','add','origin',URL],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','--depth','1','origin',REF],check=True); subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','--force','FETCH_HEAD'],check=True)
subprocess.run([sys.executable,'-m','pip','install','diffusers','transformers','accelerate','ftfy','sentencepiece','safetensors','huggingface_hub','numpy','Pillow'],check=True); subprocess.run(['ffmpeg','-version'],check=True)
print('Pinned source:', subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip())


## 生成新内容并运行全部固定条件

不依赖旧共享terminal文件。运行器将新的shared_terminal_normalized.pt、生成元数据、各前向计数、视频、观察、候选和失败保存到Drive/Video-WM/WanStateClockReplication。

条件为正常、匹配二次保存、删除90、重复90。先生成并保存正常MP4，再以相同第二次编码派生编辑和重存分支。默认执行，无参数扫描。


In [ ]:
from datetime import datetime,timezone
CONFIG=SOURCE/'runtime/tstwv2/state_clock_replication.json'; RUN_ID='wan_state_clock_replication_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); OUTPUT=Path('/content/drive/MyDrive/Video-WM/WanStateClockReplication')/RUN_ID
if OUTPUT.exists(): raise FileExistsError(OUTPUT)
import os,signal
cmd=[sys.executable,'-u','-m','runtime.tstwv2.run','--config',str(CONFIG),'--output',str(OUTPUT)]; LOG=OUTPUT.parent/f'{RUN_ID}.launcher.log'; LOG.parent.mkdir(parents=True,exist_ok=True)
with LOG.open('w') as log:
    p=subprocess.Popen(cmd,cwd=SOURCE,start_new_session=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout: print(line,end=''); log.write(line); log.flush()
        code=p.wait()
    except BaseException:
        try: p.send_signal(signal.SIGTERM)
        except ProcessLookupError: pass
        try: p.wait(timeout=5)
        except subprocess.TimeoutExpired: os.killpg(p.pid,signal.SIGKILL); p.wait()
        raise
print('launcher exit', code)
print('Results:', OUTPUT)
print('Log:', LOG)
if code: raise subprocess.CalledProcessError(code,cmd)


## 核对消息与时间对应

重点比较 local_matched（无创新项）、local_without_update（预测但不更新）、local_state（完整更新），三者使用相同路径与事件代价。global_matched/global_state保留。

事件窗口5只从删插时间指标排除，仍参与全部评分；正常时间分母11，删插10。窗口6起是名义编辑后对应，不能宣称精确定位第90帧。

两方法均正确可视为局部同步复现，但不证明更新必要；完整方法修复匹配错误才提供额外收益线索。两者不稳先查观察与歧义；更新更差则考虑修改或舍弃。


In [ ]:
import json
result=json.loads((OUTPUT/'result.json').read_text())
print(json.dumps({k:result[k] for k in ('status','source_commit','diagnostic_denominator','fixed_calls','actual_calls','failures') if k in result},ensure_ascii=False,indent=2))
for name,row in result['videos'].items():
    print(name,row['status'])
    for mode,ranking in row.get('detection',{}).get('rankings',{}).items():
        print(mode,'best=',ranking['best'],'message_unique=',ranking['message_unique'])
    print('reporting_only=',row.get('reporting_only'))
print('Full candidates/classes and observer traces:',OUTPUT/'detections')
